# Fundamentals 10 - Environment Eval API

**Fundamentals explora la API.** Este notebook muestra environments y evals sobre un batch de **5 problemas multi-operacion**.

No hay parser de lenguaje natural ni modulo aritmetico agrupado. El batch se escribe explicito para que el usuario vea que recibe `AgenticEnvironment` y que valida `run_eval`.


In [ ]:
from typing import Any

import agentic_systems as toolkit

toolkit.show({
    "checkpoint": "2.4.9.9",
    "layer": "fundamentals",
    "focus": "explorar Environment + Eval API con definiciones visibles",
    "environment_design": "Gymnasium-like without Gymnasium dependency",
})


## Escenario didactico compartido

Todos los notebooks de `tutorials/` usan este mismo problema para comparar la API sin cambiar de caso cada vez.


## Parámetros de `RunPolicy`

`RunPolicy` declara como debe comportarse una ejecucion antes de llamar al agente o al runtime. No es metadata decorativa: limita loops, define reparacion, controla trazas y hace que el resultado sea evaluable.

| Parametro | Que controla | Uso recomendado |
|---|---|---|
| `max_turns` | Numero maximo de turnos internos del agente. | Mantenerlo bajo en notebooks para evitar loops largos. |
| `max_tool_calls` | Numero maximo de llamadas a tools. | Declararlo cuando el ejercicio espera tools concretas. |
| `max_tokens` | Limite de tokens del modelo cuando el provider lo soporta. | util en providers LM; puede quedar `None` en `python-runtime`. |
| `temperature` | Aleatoriedad del modelo. | `0.0` para tutoriales reproducibles; `None` delega al provider. |
| `tool_choice` | Estrategia de seleccion de tools, por ejemplo `auto`. | `auto` cuando el agente decide; explicito cuando quieres forzar una tool. |
| `repair` | Permite reparacion automatica de salidas o tool calls invalidas. | `True` para UX robusta; `False` si quieres ver fallos crudos. |
| `max_repairs` | Maximo de intentos de reparacion. | `1` o `2` en tutoriales para mostrar control sin ocultar errores. |
| `finalize` | Que hacer al agotar turnos, por ejemplo `on_max_turns`. | Mantenerlo explicito en agentes LM evaluables. |
| `trace` | Nivel de trazabilidad (`compact`, `debug`, etc.). | `compact` para notebooks; `debug` solo para diagnostico. |
| `strict` | Si el contrato debe aplicarse de forma estricta. | `True` para ensenar API y evitar ambiguedad. |


In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

# La estructura solicitada por el usuario se materializa como datos simples.
# No es un parser ni una respuesta precocinada: solo representa la seccion `Dime:`.
REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

toolkit.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Escenario didactico  visible")


## 1) Batch de 5 casos

Cada fila es un step del episodio. El primer caso es exactamente el escenario didactico; los otros cuatro tienen entre 3 y 5 operaciones.


In [ ]:
def make_math_batch() -> list[dict[str, Any]]:
    """Devuelve 5 problemas; cada registro muestra prompt, operaciones y salidas solicitadas."""

    return [
        {
            "case_id": "math_01_default",
            "prompt": USER_PROMPT,
            "initial": 10,
            "operations": [("add", 20), ("subtract", 9), ("multiply", 4), ("divide", 2)],
            "requested_outputs": REQUESTED_OUTPUTS,
        },
        {
            "case_id": "math_02",
            "prompt": """Empieza con 100, resta 25, divide entre 5 y suma 7.
Dime:
- procedimiento
- resultado final""".strip(),
            "initial": 100,
            "operations": [("subtract", 25), ("divide", 5), ("add", 7)],
            "requested_outputs": REQUESTED_OUTPUTS,
        },
        {
            "case_id": "math_03",
            "prompt": """Empieza con 7, multiplica por 6, suma 8, resta 10 y divide entre 5.
Dime:
- procedimiento
- resultado final""".strip(),
            "initial": 7,
            "operations": [("multiply", 6), ("add", 8), ("subtract", 10), ("divide", 5)],
            "requested_outputs": REQUESTED_OUTPUTS,
        },
        {
            "case_id": "math_04",
            "prompt": """Empieza con 81, divide entre 9, suma 11, multiplica por 2 y resta 10.
Dime:
- procedimiento
- resultado final""".strip(),
            "initial": 81,
            "operations": [("divide", 9), ("add", 11), ("multiply", 2), ("subtract", 10)],
            "requested_outputs": REQUESTED_OUTPUTS,
        },
        {
            "case_id": "math_05",
            "prompt": """Empieza con 3, suma 17, multiplica por 2, resta 5, divide entre 5 y suma 1.
Dime:
- procedimiento
- resultado final""".strip(),
            "initial": 3,
            "operations": [("add", 17), ("multiply", 2), ("subtract", 5), ("divide", 5), ("add", 1)],
            "requested_outputs": REQUESTED_OUTPUTS,
        },
    ]


def render_number(value: int | float) -> int | float:
    return int(value) if isinstance(value, float) and value.is_integer() else value


def apply_operation(value: int | float, operation: str, amount: int | float) -> tuple[int | float, str]:
    previous = value
    if operation == "add":
        value = value + amount
        symbol = "+"
    elif operation == "subtract":
        value = value - amount
        symbol = "-"
    elif operation == "multiply":
        value = value * amount
        symbol = ""
    elif operation == "divide":
        if amount == 0:
            raise ValueError("No se puede dividir entre cero.")
        value = value / amount
        symbol = ""
    else:
        raise ValueError(f"Operacion no soportada: {operation!r}")
    value = render_number(value)
    return value, f"{render_number(previous)} {symbol} {render_number(amount)} = {value}"


def execute_operations(initial: int | float, operations: list[tuple[str, int | float]]) -> dict[str, Any]:
    value = initial
    procedure = []
    for operation, amount in operations:
        value, explanation = apply_operation(value, operation, amount)
        procedure.append(explanation)
    return {"procedimiento": procedure, "resultado_final": value}


cases = make_math_batch()
toolkit.show({
    "case_count": len(cases),
    "primer_prompt": cases[0]["prompt"],
    "primer_caso": cases[0],
    "all_cases": cases,
})


## 2) Definir `transition_fn` y `reward_fn`

El environment no sabe si detras hay tool, agent, multi-agent, LangGraph u OpenAI Agents. Solo ejecuta una transicion por fila y calcula reward.


In [ ]:
def arithmetic_transition(row: dict[str, Any], action: Any, info: dict[str, Any]) -> dict[str, Any]:
    """Environment transition function independiente de agentes o graphs."""

    answer = execute_operations(row["initial"], row["operations"])
    previous_memory = info.get("memory") or {}
    return {
        "selected_tool": "environment_transition",
        "summary": f"{row['case_id']}: result={answer['resultado_final']}",
        "prompt": row["prompt"],
        "procedure": answer["procedimiento"],
        "result": answer["resultado_final"],
        "requested_outputs": row["requested_outputs"],
        "respuesta_materializada": {key: answer[key] for key in row["requested_outputs"] if key in answer},
        "ok": True,
        "memory": {
            **previous_memory,
            "processed": [*previous_memory.get("processed", []), row["case_id"]],
            "last_result": answer["resultado_final"],
        },
    }


def arithmetic_reward(state: dict[str, Any], row: dict[str, Any], action: Any, env: toolkit.AgenticEnvironment) -> float:
    """Reward de 1 punto cuando la transicion termino correctamente."""

    return 1.0 if state.get("ok") else 0.0

toolkit.show({
    "transition_fn": "row.initial + row.operations -> respuesta_materializada",
    "reward_fn": "state.ok == True",
})


## 3) Construir y ejecutar Environment

`AgenticEnvironment` recorre records como episodio: `reset()` inicializa, `step()` avanza y `summary()` resume.


In [ ]:
env = toolkit.AgenticEnvironment(
    records=cases,
    name="fundamentals_environment_eval",
    transition_fn=arithmetic_transition,
    reward_fn=arithmetic_reward,
    render_mode="history",
)

observation, info = env.reset(seed=247)

while observation is not None:
    observation, reward, terminated, truncated, info = env.step()
    if terminated or truncated:
        break

toolkit.show(env.summary())


## 4) Finalize del episodio

En environment/evals, `finalize` no crea una respuesta conversacional. Cierra el episodio: toma el historial de steps y materializa un resumen auditable para usuario/evaluador.

In [ ]:
def finalize_episode(environment: toolkit.AgenticEnvironment) -> dict[str, Any]:
    cases = []
    for event in environment.history:
        graph_state = event.graph_state
        cases.append({
            "case_id": event.row.get("case_id"),
            "prompt": event.row.get("prompt"),
            "resultado_final": graph_state.get("result"),
            "procedimiento": graph_state.get("procedure"),
            "reward": event.reward,
            "ok": bool(graph_state.get("ok")),
        })

    summary = environment.summary()
    return {
        "requested_outputs": REQUESTED_OUTPUTS,
        "steps": summary["steps"],
        "passed_steps": summary["passed_steps"],
        "failed_steps": summary["failed_steps"],
        "total_reward": summary["total_reward"],
        "cases": cases,
    }


episode_final = finalize_episode(env)
toolkit.show(episode_final, title="Finalize  episode outputs")

## 5) Lineage Memory del episodio

Aqui se ve que paso paso a paso: decisiones, reward y evidencia minima.


In [ ]:
lineage = env.lineage(
    question="Procesa 5 casos aritmeticos multi-operacion; el primero es el escenario didactico de fundamentals.",
    goal="Explicar el episodio environment/eval sin depender de un framework externo.",
    tags=["fundamentals", "environment", "eval"],
)

toolkit.show(lineage)
toolkit.show({"compact_context": lineage.to_prompt_context(max_chars=1200)})


## 6) Definir tools, contrato, policy y agente de eval

`run_eval` evalua un agente sobre casos declarativos. Para mantenerlo local y determinista, este agente usa `python-runtime`.

Lo profesional aqui es declarar contrato y policy arriba; despues `toolkit.agent(...)` solo recibe objetos ya nombrados.

In [ ]:
@toolkit.tool
def add(a: float, b: float) -> dict:
    return {"value": a + b}


@toolkit.tool
def subtract(a: float, b: float) -> dict:
    return {"value": a - b}


@toolkit.tool
def multiply(a: float, b: float) -> dict:
    return {"value": a * b}


@toolkit.tool
def divide(a: float, b: float) -> dict:
    return {"value": a / b}


eval_tools = [add, subtract, multiply, divide]

eval_contract = toolkit.AgentContract(
    tool_expectation=toolkit.expect.any_of("add", "subtract", "multiply", "divide"),
)

eval_policy = toolkit.RunPolicy(
    max_tool_calls=1,
    temperature=0.0,
    trace="compact",
)

eval_agent = toolkit.agent(
    name="fundamentals_eval_agent",
    instructions="Ejecuta una operaci?n estructurada usando las tools disponibles.",
    tools=eval_tools,
    engine="python-runtime",
    contract=eval_contract,
    policy=eval_policy,
)

toolkit.show({
    "eval_agent": eval_agent.info(),
    "eval_tools": [tool.name for tool in eval_tools],
    "eval_contract": eval_contract.model_dump(mode="json"),
    "eval_policy": eval_policy.model_dump(mode="json"),
})

## 7) Construir casos de eval desde el mismo batch

Para mantener simple el ejemplo, cada caso de eval valida la **primera operacion** de cada problema. El environment ya valida el problema completo multi-operacion.


In [ ]:
eval_cases = []
for row in cases:
    first_op, amount = row["operations"][0]
    expected_first = execute_operations(row["initial"], [(first_op, amount)])["resultado_final"]
    eval_cases.append({
        "name": row["case_id"],
        "input": {"tool": first_op, "input": {"a": row["initial"], "b": amount}},
        "expected": {
            "must_call": [first_op],
            "data_contains": {"tool": first_op, "ok": True, "value": expected_first},
        },
    })

toolkit.show({
    "eval_cases": eval_cases,
    "nota": "Los expected de eval se calculan desde la primera operacion declarada en cada record.",
})


## 8) Ejecutar run_eval con reproducibilidad explicita

El agente usa python-runtime y fixtures locales, por lo que esta evaluacion se
clasifica como determinista. La clasificacion describe condiciones observables;
no convierte Providers LM en deterministas.

In [ ]:
report = toolkit.run_eval(
    eval_agent,
    eval_cases,
    determinism="deterministic",
    seed=247,
    reproducibility_conditions=[
        "same local fixtures",
        "python-runtime",
        "same Agent contract and policy",
    ],
)
toolkit.show(report.to_dict())

report_lineage = report.lineage(
    name="fundamentals.arithmetic.eval.lineage",
    question="Evalua 5 decisiones aritmeticas con python-runtime.",
    goal="Mostrar que evals y environments comparten batch, validacion y lineage.",
)
toolkit.show(report_lineage)

toolkit.human_result(
    title="Eval batch + Lineage Memory  fundamentals",
    result=report,
    pretty=True,
    show_lineage=True,
    lineage=report_lineage,
)


## Lo importante

- Environment procesa episodios por step.
- `finalize_episode(...)` cierra el episodio desde `env.history`, no desde un texto libre.
- Eval reutiliza el patron batch, pero aqui validamos un subproblema para mantener el demo corto.
- Lineage Memory compacta lo ocurrido sin reenviar todo el historial bruto.
- El escenario didactico queda incluido como `math_01_default`.
- EvalReport.reproducibility registra clasificacion, seed y condiciones.


## Coverage API de este notebook

Esta tabla deja explicito que parte de Agentic Systems queda materializada aqui.


In [ ]:
api_coverage = [
    {
        "api": "AgenticEnvironment",
        "description": "Materializa un entorno con transicion, reward y episodios."
    },
    {
        "api": "records",
        "description": "Declara los casos del entorno como datos, no como narrativa oculta."
    },
    {
        "api": "transition_fn",
        "description": "Define como cambia el estado paso a paso."
    },
    {
        "api": "reward_fn",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "env.reset",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "env.step",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "env.summary",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "finalize_episode",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "env.lineage",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "@toolkit.tool",
        "description": "Construye herramientas declarativas para reutilizarlas en agentes y grafos."
    },
    {
        "api": "toolkit.agent(engine='python-runtime')",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "run_eval",
        "description": "Ejecuta evals sobre el mismo dominio que el environment."
    },
    {
        "api": "EvalReport.reproducibility",
        "description": "Registra clasificacion y condiciones de replay."
    },
    {
        "api": "EvalReport.lineage",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "lineage.to_prompt_context",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "5-case arithmetic batch",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "default case included",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    }
]

toolkit.show({"notebook": "10_environment_eval_api.ipynb", "api_coverage": api_coverage})

## Simbolos API explicados

Este notebook se alinea con `docs/API.md` y ensena estos simbolos publicos:

- `AgenticEnvironment`: Environment episidico con transition y reward.
- `EnvironmentTransition / EpisodeResult`: Tipos publicos de environment.
- `environment_lineage`: Lineage publico para episodios.
- `run_eval / Evaluator / EvalCaseResult / EvalReport`: API publica de evaluacion.
- `environment_summary / eval_report_output / eval_report_summary`: Vistas publicas de environment/evals.

